$
\begin{equation}
\frac{\partial \xi_+}{\partial n(z_i)} \approx \frac{\xi_+[\theta;~n(z)+\epsilon \delta(z-z_i)]-\xi_+[\theta;~n(z)]}{\epsilon}
\end{equation}
$

In [1]:
import numpy as np
import cosmolike_lsst_y1_interface as ci
# import euclidemu2
import getdist
import os
import camb
import scipy
import itertools
import iminuit
import functools
import matplotlib.pyplot as plt
import cocoa_photoz as cp

# Libs to explore the derivative of xi_+ wrt n(z)
from scipy.signal import unit_impulse
from multiprocessing import Pool

In [2]:
path = "../../../external_modules/data/lsst_y1/lsst_y1_source.nz"
nz_fid = np.genfromtxt(path)
(theta_fid, xip_fid, xim_fid) = \
    cp.xi(external_nz_modeling=1,mod_nz=nz_fid)

In [29]:
epsilon = 0.01
tomo_bin = 1
n_z = nz_fid.shape[0]
z_vals = nz_fid[:,0]

def compute_derivative(args):
    z_idx = args
    nz_per = nz_fid.copy()
    nz_per[z_idx, tomo_bin] += epsilon
    nz_per[:,tomo_bin] /= np.trapz(y=nz_per[:,tomo_bin], x=z_vals)
    (theta_per, xip_per, xim_per) = \
        cp.xi(external_nz_modeling=1,mod_nz=nz_per)
    dxi_dn = (xip_per[:,0,0] - xip_fid[:,0,0]) / epsilon
    print(z_idx)
    return dxi_dn

jobs = list(range(n_z))

In [39]:
n_cores=8
with Pool(processes=n_cores) as pool:
    results = pool.map(compute_derivative, jobs)

In [ ]:
theta_xipm = []

for eps in [0.000001,0.0001,0.01,0.2,0.4,1]:
    print(eps)
    (theta_per, xip_per, xim_per) = \
        cp.xi(external_nz_modeling=1,mod_nz=lsst_y1_source_nz(fiducial_nz=False,eps=eps,eps_loc=0))
    theta_xipm.append([(theta_per, xip_per, xim_per)])

In [ ]:
## deriv vs epsilon
ti,tj=0,0
color = ["k","C0","r","gray","purple","orange"]
for eps_idx, (c,eps) in enumerate(zip(color,[0.000001,0.0001,0.01,0.2,0.4,1])):
    print(eps)
    theta_per, xip_per, xim_per = theta_xipm[eps_idx][0]
    deriv = ( xip_per[:,ti,tj] - xip_fid[:,ti,tj] ) / eps
    for theta_idx in range(len(theta_fid)):    
        plt.plot(eps, deriv[theta_idx],"o",label=f"ϵ = {eps}" if theta_idx==0 else None,color=c)

plt.legend(loc="best")